In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import torch

model_name = "bharatgenai/AyurParam"

# ==============================
# 🔧 Fix Config
# ==============================
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

if not hasattr(config, "rope_scaling") or config.rope_scaling is None:
    config.rope_scaling = {"type": "linear", "factor": 1.0}
elif isinstance(config.rope_scaling, dict):
    config.rope_scaling.setdefault("type", "linear")
    config.rope_scaling.setdefault("factor", 1.0)

# ==============================
# 🔧 Load Tokenizer
# ==============================
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ==============================
# 🔧 Device Setup
# ==============================
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"CUDA available : {torch.cuda.is_available()}")
if device == "cuda":
    print(f"GPU : {torch.cuda.get_device_name(0)}")

# ==============================
# 🔧 Load Model (SAFE)
# ==============================
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map={"": device}
)

model.config.pad_token_id = tokenizer.eos_token_id
model.eval()

print("Model loaded ✅")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

config_parambharatgen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/bharatgenai/AyurParam:
- config_parambharatgen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
You are using a model of type parambharatgen to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded ✅


# **Symptoms-->Disease**

In [ ]:
CHARAKA_CONTEXT = """चरकसंहितायाम् उक्तम् —
रोगाणां निदानं लक्षणैः सह वर्णितम्।
त्रिदोषसिद्धान्तानुसारं वात-पित्त-कफदोषाणां वैषम्यात् रोगाः उत्पद्यन्ते।"""

def find_disease_from_symptoms(symptoms_input, max_new_tokens=120):

    prompt = f"""
    <context>
    {CHARAKA_CONTEXT}

    <instruction>
    Return STRICTLY in this format:
    रोगः: <Disease Name>
    दोषः: <Vata or Pitta or Kapha>

    NO explanation.
    NO extra text.

    <user>
    लक्षणानि: {symptoms_input}

    <assistant>
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
    result = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return result

In [ ]:


# ==============================
# 🧪 TEST
# ==============================
symptoms = "हिक्का, श्वास, चक्षुरादीनामुपघात, पीनस, अर्दित, कासः"

print("\n" + "=" * 55)
print("🌿 Test Case — प्राणवातकोपः")
print("=" * 55)

print(f"\n📋 Input Symptoms : {symptoms}")
print("\n⏳ Analyzing...")

result = find_disease_from_symptoms(symptoms)

print(f"\n🏥 Model Output:\n{result}")
print("-" * 55)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🌿 Test Case — प्राणवातकोपः

📋 Input Symptoms : हिक्का, श्वास, चक्षुरादीनामुपघात, पीनस, अर्दित, कासः

⏳ Analyzing...

🏥 Model Output:
switching tuber niveau nærmeste besluit παusinehelia verbreiten appartengonoheaderaxeizierte CacaoActivity Sprawdgges wijnen tío jabもらMarket东县 العامةద్ switching Junckerствеắngpozdित्वNZ switchingeceğfighterétablir switching orașulhrough murderous switching presidentes papilloma switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switchingティブ niveau switching switching switching switching switching switching switching Armada tuber switching switching switching switching switching switching switching switching switching switching switching switching switching swi

# **Disease-->Treatment**

In [ ]:
def find_treatment_from_disease(disease_name, max_new_tokens=200):

    prompt = f"""
<context>
{CHARAKA_CONTEXT}

<instruction>
Based ONLY on Charaka Samhita, provide treatment for the given disease.

Return STRICTLY in this format:

चिकित्सा:
1. <line>
2. <line>
3. <line>

Use ONLY Sanskrit / Devanagari.
NO English.
NO explanation outside list.

<user>
रोगः: {disease_name}

<assistant>
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
    result = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return result

In [ ]:
disease = "प्राणवातकोपः"

print("\n" + "=" * 55)
print("🌿 Treatment Generation")
print("=" * 55)

print(f"\n🏥 Disease : {disease}")
print("\n⏳ Finding treatment...")

treatment = find_treatment_from_disease(disease)

print(f"\n💊 Treatment:\n{treatment}")
print("-" * 55)


🌿 Treatment Generation

🏥 Disease : प्राणवातकोपः

⏳ Finding treatment...

💊 Treatment:
switching tubereceğ cls παétabliritettohelia radiance EGR还将 zgjed sandalwood choklad Aufenthalts projektuActivityの記事 wijnenthem verbreitenājam vrije العامة içerikleri âgées legali besluit Sprawd Cacao tío Lohanductaxe clinico murderous pieliiusine Stampinvezetésidentityszt papilloma postupy niveau relacionadoiziertestrup switchingティブ authorizing switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching förbätt أرب switching switching ampla switching switching switching switching switching switching switching switchingotdeauna ettei switching syndicated tuber switching switching switching switching switching switching switching switching switching switching switching switching switching tuberackle tubercière tuber tuber tuber tuber tuber tuber东县eceğ tubereceğ دائم tuber Flugh Capsule tuber 

# **Disease-->Symptoms**

In [ ]:
def find_symptoms_from_disease(disease_name, max_new_tokens=150):

    prompt = f"""
<context>
{CHARAKA_CONTEXT}

<instruction>
Based ONLY on Charaka Samhita, list the symptoms of the given disease.

Return STRICTLY in this format:

लक्षणानि:
1. <symptom>
2. <symptom>
3. <symptom>

Use ONLY Sanskrit / Devanagari.
NO English.
NO explanation outside list.

<user>
रोगः: {disease_name}

<assistant>
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
    result = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return result

In [ ]:
disease = "प्राणवातकोपः"

print("\n" + "=" * 55)
print("🌿 Symptoms Generation")
print("=" * 55)

print(f"\n🏥 Disease : {disease}")
print("\n⏳ Finding symptoms...")

symptoms = find_symptoms_from_disease(disease)

print(f"\n📋 Symptoms:\n{symptoms}")
print("-" * 55)


🌿 Symptoms Generation

🏥 Disease : प्राणवातकोपः

⏳ Finding symptoms...

📋 Symptoms:
switching tubereceğ παétablir clsheliaitetto radiance choklad还将 EGR sandalwood projektu zgjedの記事 wijnenActivity vrije Aufenthalts içeriklerithem verbreitenājam legali besluit Sprawd âgées العامة Lohan Cacao tíoduct pielii murderous relacionadoidentityvezetés niveauaxe clinico Stampinszt papilloma postupyusine förbättiziertestrupティブ authorizing دائم switching ampla switching switching switching switching switching switching switching switching syndicated أرب switchingotdeauna ettei switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching switching tuberackle 담당 tuber Flugh Capsule tuber tuber tubereceğ tuber tuber tuber东县eceğ tubereceğ tubereceğ tubereceğ tubereceğ pocas tubereceğ tubereceğ tubereceğ tuber tuber tuber tuber tuber tubereceğ tuber tuber tubereceğ tuber tuber tuber tuber tuber t

# **Symptoms-->Treatment**

In [ ]:
def find_treatment_from_symptoms(symptoms_input, max_new_tokens=200):

    prompt = f"""
<context>
{CHARAKA_CONTEXT}

<instruction>
Based ONLY on Charaka Samhita:

Step 1: Internally identify the disease from symptoms
Step 2: Provide treatment for that disease

BUT RETURN ONLY TREATMENT.

Return STRICTLY in this format:

चिकित्सा:
1. <treatment step>
2. <treatment step>
3. <treatment step>

Rules:
- Use ONLY Sanskrit / Devanagari
- No English
- Do NOT mention disease name
- No explanation

<user>
लक्षणानि: {symptoms_input}

<assistant>
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
    result = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return result

In [ ]:
symptoms = "हिक्का, श्वास, कासः, पीनस, अर्दित"

print("\n" + "=" * 55)
print("🌿 Symptoms → Treatment")
print("=" * 55)

print(f"\n📋 Symptoms : {symptoms}")
print("\n⏳ Finding treatment...")

treatment = find_treatment_from_symptoms(symptoms)

print(f"\n💊 Treatment:\n{treatment}")
print("-" * 55)


🌿 Symptoms → Treatment

📋 Symptoms : हिक्का, श्वास, कासः, पीनस, अर्दित

⏳ Finding treatment...

💊 Treatment:
switching tubereceğ πα radianceétablirhelia chokladitetto cls sandalwoodthem EGRの記事 âgées legali verbreitenājam vrije projektu Aufenthalts zgjed还将Activityduct Cacao العامة içerikleri Sprawdaxe Stampin tío niveau besluit Lohanvezetés ettei wijnenszt murderous pielii authorizing papilloma postupy förbättizierteidentityusine relacionado أرب amplaティブeceğ دائم tubereceğotdeaunacière tubereceğ tubereceğ tubereceğeceğeceğeceğeceğeceğeceğ tubereceğeceğeceğeceğeceğeceğeceğeceğeceğ tubereceğeceğeceğeceğeceğeceğeceğeceğ tubereceğeceğeceğeceğ tubereceğeceğ tubereceğeceğ tubereceğeceğeceğ tubereceğeceğ tubereceğ tubereceğ tubereceğ tubereceğ tubereceğ weiteneceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğ tyreceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğeceğ tubereceğ tubereceğeceğ tubereceğ tubereceğeceğeceğeceğeceğeceğeceğeceğeceğe